# CAGP on YAGO3-10 (Medium Diversity)

**Goal:** Validate CAGP on medium-diversity KG.

**Dataset:** YAGO3-10 (37 relations)

**Expected:** CAGP ≥ VanillaGPKGE (both should work, CAGP may be slightly better)

In [ ]:
!pip install -q torch numpy scikit-learn matplotlib pykeen

In [ ]:
# Configuration
# QUICK MODE: For fast validation (~5 min on CPU)
QUICK_MODE = True

if QUICK_MODE:
    CONFIG = {
        'epochs': 15,
        'embedding_dim': 50,
        'batch_size': 2048,
        'lr': 0.001,
        'kl_weight': 0.01,
        'seeds': [42],  # Single seed
    }
    print("⚡ QUICK MODE: Single seed, reduced epochs")
else:
    # FULL MODE: For paper submission (run on Colab GPU)
    CONFIG = {
        'epochs': 50,
        'embedding_dim': 100,
        'batch_size': 1024,
        'lr': 0.001,
        'kl_weight': 0.01,
        'seeds': [42, 123, 456],
    }
    print("📊 FULL MODE: 3 seeds, 50 epochs")

In [ ]:
CONFIG = {
    'epochs': 50,
    'embedding_dim': 100,
    'batch_size': 1024,
    'lr': 0.001,
    'kl_weight': 0.01,
    'seeds': [42, 123, 456],
}

In [ ]:
def load_yago310():
    from pykeen.datasets import YAGO310
    dataset = YAGO310(create_inverse_triples=False)
    
    def extract_triples(tf):
        return [(h, r, t) for h, r, t in tf.triples]
    
    train = extract_triples(dataset.training)
    valid = extract_triples(dataset.validation)
    test = extract_triples(dataset.testing)
    
    entities = set()
    relations = set()
    for h, r, t in train + valid + test:
        entities.add(h)
        entities.add(t)
        relations.add(r)
    
    return {
        'train': train, 'valid': valid, 'test': test,
        'entities': entities, 'relations': relations,
    }

data = load_yago310()
print(f"YAGO3-10: {len(data['train'])} train, {len(data['test'])} test")
print(f"Entities: {len(data['entities'])}, Relations: {len(data['relations'])}")

In [ ]:
class DistMult(nn.Module):
    def __init__(self, num_entities, num_relations, dim):
        super().__init__()
        self.entity_emb = nn.Embedding(num_entities, dim)
        self.relation_emb = nn.Embedding(num_relations, dim)
        nn.init.xavier_uniform_(self.entity_emb.weight)
        nn.init.xavier_uniform_(self.relation_emb.weight)

    def forward(self, heads, relations, tails):
        h = self.entity_emb(heads)
        r = self.relation_emb(relations)
        t = self.entity_emb(tails)
        return (h * r * t).sum(dim=-1)

    def get_uncertainty(self, heads, relations, tails):
        scores = torch.sigmoid(self.forward(heads, relations, tails))
        uncertainty = -scores * torch.log(scores + 1e-10) - (1-scores) * torch.log(1-scores + 1e-10)
        return uncertainty


class VanillaGPKGE(nn.Module):
    def __init__(self, num_entities, num_relations, dim):
        super().__init__()
        self.num_entities = num_entities
        self.entity_mean = nn.Parameter(torch.randn(num_entities, dim) * 0.1)
        self.entity_logvar = nn.Parameter(torch.zeros(num_entities, dim) - 1.0)
        self.relation_emb = nn.Embedding(num_relations, dim)
        nn.init.xavier_uniform_(self.relation_emb.weight)

    def forward(self, heads, relations, tails, use_sampling=True):
        if use_sampling and self.training:
            h = self._sample(heads)
            t = self._sample(tails)
        else:
            h = self.entity_mean[heads]
            t = self.entity_mean[tails]
        r = self.relation_emb(relations)
        return (h * r * t).sum(dim=-1)

    def _sample(self, indices):
        mean = self.entity_mean[indices]
        std = torch.exp(0.5 * self.entity_logvar[indices])
        return mean + std * torch.randn_like(std)

    def get_uncertainty(self, heads, relations, tails):
        h_var = torch.exp(self.entity_logvar[heads]).mean(dim=-1)
        t_var = torch.exp(self.entity_logvar[tails]).mean(dim=-1)
        return (h_var + t_var) / 2

    def kl_loss(self):
        kl = -0.5 * torch.sum(1 + self.entity_logvar - self.entity_mean.pow(2) - self.entity_logvar.exp())
        return kl / self.num_entities


class CoverageAugmentedGPKGE(nn.Module):
    def __init__(self, num_entities, num_relations, dim, initial_alpha=0.5, learn_alpha=True):
        super().__init__()
        self.num_entities = num_entities
        self.num_relations = num_relations
        
        self.entity_mean = nn.Parameter(torch.randn(num_entities, dim) * 0.1)
        self.entity_logvar = nn.Parameter(torch.zeros(num_entities, dim) - 1.0)
        self.relation_emb = nn.Embedding(num_relations, dim)
        nn.init.xavier_uniform_(self.relation_emb.weight)
        
        self.register_buffer('coverage', torch.zeros(num_entities, num_relations))
        
        if learn_alpha:
            self.alpha_logit = nn.Parameter(torch.logit(torch.tensor(initial_alpha)))
        else:
            self.register_buffer('alpha_logit', torch.logit(torch.tensor(initial_alpha)))

    def forward(self, heads, relations, tails, use_sampling=True):
        if use_sampling and self.training:
            h = self._sample(heads)
            t = self._sample(tails)
        else:
            h = self.entity_mean[heads]
            t = self.entity_mean[tails]
        r = self.relation_emb(relations)
        return (h * r * t).sum(dim=-1)

    def _sample(self, indices):
        mean = self.entity_mean[indices]
        std = torch.exp(0.5 * self.entity_logvar[indices])
        return mean + std * torch.randn_like(std)

    def get_uncertainty(self, heads, relations, tails):
        h_var = torch.exp(self.entity_logvar[heads]).mean(dim=-1)
        t_var = torch.exp(self.entity_logvar[tails]).mean(dim=-1)
        gp_var = (h_var + t_var) / 2
        
        h_seen = self.coverage[heads, relations]
        t_seen = self.coverage[tails, relations]
        coverage_unc = 2.0 - h_seen - t_seen
        
        gp_var_norm = gp_var / (gp_var.mean() + 1e-8) * (coverage_unc.mean() + 1e-8)
        
        alpha = torch.sigmoid(self.alpha_logit)
        return alpha * gp_var_norm + (1 - alpha) * coverage_unc

    def precompute_coverage(self, triples, entity_to_idx, relation_to_idx):
        for h, r, t in triples:
            self.coverage[entity_to_idx[h], relation_to_idx[r]] = 1.0
            self.coverage[entity_to_idx[t], relation_to_idx[r]] = 1.0

    def kl_loss(self):
        kl = -0.5 * torch.sum(1 + self.entity_logvar - self.entity_mean.pow(2) - self.entity_logvar.exp())
        return kl / self.num_entities
    
    def get_alpha(self):
        return torch.sigmoid(self.alpha_logit).item()


print("Models defined")

In [ ]:
def train_model(model, triples, entity_to_idx, relation_to_idx, epochs, is_gp=False):
    model = model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=CONFIG['lr'])
    criterion = nn.BCEWithLogitsLoss()

    if hasattr(model, 'precompute_coverage'):
        model.precompute_coverage(triples, entity_to_idx, relation_to_idx)

    heads = torch.tensor([entity_to_idx[h] for h, r, t in triples])
    relations = torch.tensor([relation_to_idx[r] for h, r, t in triples])
    tails = torch.tensor([entity_to_idx[t] for h, r, t in triples])

    num_entities = len(entity_to_idx)
    dataset = TensorDataset(heads, relations, tails)
    loader = DataLoader(dataset, batch_size=CONFIG['batch_size'], shuffle=True)

    model.train()
    for epoch in range(epochs):
        for batch_h, batch_r, batch_t in loader:
            batch_h, batch_r, batch_t = batch_h.to(device), batch_r.to(device), batch_t.to(device)

            if is_gp:
                pos_scores = model(batch_h, batch_r, batch_t, use_sampling=True)
            else:
                pos_scores = model(batch_h, batch_r, batch_t)

            neg_t = torch.randint(0, num_entities, batch_t.shape, device=device)
            if is_gp:
                neg_scores = model(batch_h, batch_r, neg_t, use_sampling=True)
            else:
                neg_scores = model(batch_h, batch_r, neg_t)

            loss = criterion(pos_scores, torch.ones_like(pos_scores)) + \
                   criterion(neg_scores, torch.zeros_like(neg_scores))

            if is_gp and hasattr(model, 'kl_loss'):
                loss += CONFIG['kl_weight'] * model.kl_loss()

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

    return model


def evaluate_auroc(model, test_triples, entity_to_idx, relation_to_idx):
    model.eval()
    heads = torch.tensor([entity_to_idx.get(h, 0) for h, r, t in test_triples]).to(device)
    relations = torch.tensor([relation_to_idx.get(r, 0) for h, r, t in test_triples]).to(device)
    tails = torch.tensor([entity_to_idx.get(t, 0) for h, r, t in test_triples]).to(device)

    with torch.no_grad():
        id_unc = model.get_uncertainty(heads, relations, tails).cpu().numpy()
        neg_tails = torch.randint(0, len(entity_to_idx), tails.shape, device=device)
        ood_unc = model.get_uncertainty(heads, relations, neg_tails).cpu().numpy()

    labels = np.concatenate([np.ones(len(id_unc)), np.zeros(len(ood_unc))])
    scores = np.concatenate([-id_unc, -ood_unc])
    return roc_auc_score(labels, scores)

In [ ]:
ent2idx = {e: i for i, e in enumerate(data['entities'])}
rel2idx = {r: i for i, r in enumerate(data['relations'])}

results = {'DistMult': [], 'VanillaGPKGE': [], 'CAGP': []}
alphas = []

for seed in CONFIG['seeds']:
    print(f"\n{'='*50}")
    print(f"Seed {seed}")
    print('='*50)
    
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    
    # DistMult
    dm = DistMult(len(ent2idx), len(rel2idx), CONFIG['embedding_dim'])
    dm = train_model(dm, data['train'], ent2idx, rel2idx, CONFIG['epochs'], is_gp=False)
    auroc_dm = evaluate_auroc(dm, data['test'], ent2idx, rel2idx)
    results['DistMult'].append(auroc_dm)
    print(f"  DistMult AUROC: {auroc_dm:.4f}")
    
    # Vanilla GP-KGE
    gp = VanillaGPKGE(len(ent2idx), len(rel2idx), CONFIG['embedding_dim'])
    gp = train_model(gp, data['train'], ent2idx, rel2idx, CONFIG['epochs'], is_gp=True)
    auroc_gp = evaluate_auroc(gp, data['test'], ent2idx, rel2idx)
    results['VanillaGPKGE'].append(auroc_gp)
    print(f"  VanillaGPKGE AUROC: {auroc_gp:.4f}")
    
    # CAGP
    cagp = CoverageAugmentedGPKGE(len(ent2idx), len(rel2idx), CONFIG['embedding_dim'])
    cagp = train_model(cagp, data['train'], ent2idx, rel2idx, CONFIG['epochs'], is_gp=True)
    auroc_cagp = evaluate_auroc(cagp, data['test'], ent2idx, rel2idx)
    results['CAGP'].append(auroc_cagp)
    alphas.append(cagp.get_alpha())
    print(f"  CAGP AUROC: {auroc_cagp:.4f} (α={cagp.get_alpha():.3f})")

In [ ]:
print("\n" + "="*60)
print("YAGO3-10 RESULTS (37 relations)")
print("="*60)
print(f"{'Model':<20} {'AUROC':<15} {'vs VanillaGPKGE'}")
print("-"*60)

gp_mean = np.mean(results['VanillaGPKGE'])
for model_name in ['DistMult', 'VanillaGPKGE', 'CAGP']:
    mean = np.mean(results[model_name])
    std = np.std(results[model_name])
    delta = mean - gp_mean
    print(f"{model_name:<20} {mean:.4f} ± {std:.3f}   {delta:+.4f}")

print("-"*60)
print(f"Learned α = {np.mean(alphas):.3f} ± {np.std(alphas):.3f}")

In [ ]:
import json
output = {
    'dataset': 'YAGO3-10',
    'num_relations': len(data['relations']),
    'results': {m: {'mean': float(np.mean(results[m])), 'std': float(np.std(results[m]))} for m in results},
    'learned_alpha': {'mean': float(np.mean(alphas)), 'std': float(np.std(alphas))}
}
with open('cagp_yago310_results.json', 'w') as f:
    json.dump(output, f, indent=2)
print("Results saved")